In [1]:
import pandas as pd

# Load the dataset to inspect it again
df = pd.read_csv('heart_disease_uci.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 920 entries, 0 to 919
Data columns (total 16 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        920 non-null    int64  
 1   age       920 non-null    int64  
 2   sex       920 non-null    object 
 3   dataset   920 non-null    object 
 4   cp        920 non-null    object 
 5   trestbps  861 non-null    float64
 6   chol      890 non-null    float64
 7   fbs       830 non-null    object 
 8   restecg   918 non-null    object 
 9   thalch    865 non-null    float64
 10  exang     865 non-null    object 
 11  oldpeak   858 non-null    float64
 12  slope     611 non-null    object 
 13  ca        309 non-null    float64
 14  thal      434 non-null    object 
 15  target    920 non-null    int64  
dtypes: float64(5), int64(3), object(8)
memory usage: 115.1+ KB


In [2]:
from sklearn.preprocessing import LabelEncoder

# Drop irrelevant columns
df = df.drop(columns=['id', 'dataset'])

# Rename 'num' to 'target'
df.rename(columns={'num': 'target'}, inplace=True)

# Fill missing numerical values with median
for col in ['trestbps', 'chol', 'thalch', 'oldpeak', 'ca']:
    df[col] = df[col].fillna(df[col].median())

# Fill missing categorical values with mode
for col in ['fbs', 'restecg', 'exang', 'slope', 'thal']:
    df[col] = df[col].fillna(df[col].mode()[0])

# Encode categorical columns
categorical_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# Final check for missing values
missing_values = df.isnull().sum()

# Dataset ready summary
df_ready_info = df.info()
df.head(), missing_values

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 920 entries, 0 to 919
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       920 non-null    int64  
 1   sex       920 non-null    int64  
 2   cp        920 non-null    int64  
 3   trestbps  920 non-null    float64
 4   chol      920 non-null    float64
 5   fbs       920 non-null    int64  
 6   restecg   920 non-null    int64  
 7   thalch    920 non-null    float64
 8   exang     920 non-null    int64  
 9   oldpeak   920 non-null    float64
 10  slope     920 non-null    int64  
 11  ca        920 non-null    float64
 12  thal      920 non-null    int64  
 13  target    920 non-null    int64  
dtypes: float64(5), int64(9)
memory usage: 100.8 KB


C:\Users\user\AppData\Local\Temp\ipykernel_6516\3305286873.py:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(df[col].mode()[0])


(   age  sex  cp  trestbps   chol  fbs  restecg  thalch  exang  oldpeak  slope  \
 0   63    1   3     145.0  233.0    1        0   150.0      0      2.3      0   
 1   67    1   0     160.0  286.0    0        0   108.0      1      1.5      1   
 2   67    1   0     120.0  229.0    0        0   129.0      1      2.6      1   
 3   37    1   2     130.0  250.0    0        1   187.0      0      3.5      0   
 4   41    0   1     130.0  204.0    0        0   172.0      0      1.4      2   
 
     ca  thal  target  
 0  0.0     0       0  
 1  3.0     1       2  
 2  2.0     2       1  
 3  0.0     1       0  
 4  0.0     1       0  ,
 age         0
 sex         0
 cp          0
 trestbps    0
 chol        0
 fbs         0
 restecg     0
 thalch      0
 exang       0
 oldpeak     0
 slope       0
 ca          0
 thal        0
 target      0
 dtype: int64)

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score
import joblib

# 1. Prepare your data
X = df.drop('target', axis=1)
y = df['target']

# 2. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 3. Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save the scaler for Streamlit app use
joblib.dump(scaler, 'scaler.pkl')

# 4. Define Models
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, learning_rate=0.1, use_label_encoder=False, eval_metric='mlogloss')
}

# 5. Train, Predict, Evaluate and Save Models
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    
    preds = model.predict(X_test_scaled)
    
    # Get full class probabilities (multi-class)
    probas = model.predict_proba(X_test_scaled)

    print(f"\n🧠 {name} Results:")
    print(classification_report(y_test, preds))
    
    # ROC-AUC for multi-class (One-vs-Rest)
    print("ROC-AUC Score:", roc_auc_score(y_test, probas, multi_class='ovr', average='macro'))

    # Save model
    model_filename = f"{name.replace(' ', '_').lower()}_model.pkl"
    joblib.dump(model, model_filename)



🧠 Random Forest Results:
              precision    recall  f1-score   support

           0       0.79      0.87      0.83        82
           1       0.52      0.62      0.56        53
           2       0.17      0.09      0.12        22
           3       0.18      0.14      0.16        21
           4       0.00      0.00      0.00         6

    accuracy                           0.59       184
   macro avg       0.33      0.34      0.33       184
weighted avg       0.54      0.59      0.56       184

ROC-AUC Score: 0.8079582624139334

🧠 Gradient Boosting Results:
              precision    recall  f1-score   support

           0       0.79      0.85      0.82        82
           1       0.50      0.51      0.50        53
           2       0.29      0.23      0.26        22
           3       0.11      0.10      0.10        21
           4       0.00      0.00      0.00         6

    accuracy                           0.57       184
   macro avg       0.34      0.34      0.

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:183: UserWarning: [20:25:19] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



🧠 XGBoost Results:
              precision    recall  f1-score   support

           0       0.80      0.87      0.83        82
           1       0.53      0.55      0.54        53
           2       0.47      0.36      0.41        22
           3       0.29      0.24      0.26        21
           4       0.33      0.33      0.33         6

    accuracy                           0.62       184
   macro avg       0.48      0.47      0.47       184
weighted avg       0.61      0.62      0.61       184

ROC-AUC Score: 0.8050947201636245
